# 01 — Análise exploratória médica

Visão epidemiológica da coorte hospitalar (`data/Obesity.csv`) para apoiar a equipe clínica.
O dicionário completo está em `documentacao/04-dicionario-dados.md`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "Obesity.csv").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_pipeline import LABEL_PT, add_clinical_features, load_raw_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

df = add_clinical_features(load_raw_dataset(ROOT / "data" / "Obesity.csv"))
df["Obesity_PT"] = df["Obesity"].map(LABEL_PT)
print(df.shape)
df.head()


## Qualidade e distribuição do alvo

In [ ]:
print("Nulos por coluna:\n", df.isna().sum())
print("\nDistribuição do alvo:")
print(df["Obesity_PT"].value_counts())
print("\nEstatísticas numéricas:")
df.describe().T


In [ ]:
order = list(LABEL_PT.values())
plt.figure(figsize=(9, 4))
sns.countplot(data=df, x="Obesity_PT", order=order, hue="Obesity_PT", legend=False)
plt.xticks(rotation=25, ha="right")
plt.title("Distribuição dos níveis de obesidade")
plt.xlabel("")
plt.tight_layout()


## Insights clínicos prioritários

Três eixos que o painel do Streamlit reproduz: histórico familiar, atividade física e consumo calórico.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.countplot(data=df, x="Obesity_PT", hue="family_history", order=order, ax=axes[0])
axes[0].set_title("Histórico familiar")
axes[0].tick_params(axis="x", rotation=25)
sns.boxplot(data=df, x="Obesity_PT", y="FAF", order=order, ax=axes[1])
axes[1].set_title("Atividade física (FAF)")
axes[1].tick_params(axis="x", rotation=25)
sns.countplot(data=df, x="Obesity_PT", hue="FAVC", order=order, ax=axes[2])
axes[2].set_title("Alimentos calóricos (FAVC)")
axes[2].tick_params(axis="x", rotation=25)
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()


## IMC como métrica clínica de apoio

In [ ]:
print(df["IMC"].describe())
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x="Obesity_PT", y="IMC", order=order)
plt.xticks(rotation=25, ha="right")
plt.title("IMC por nível de obesidade")
plt.xlabel("")
plt.tight_layout()


In [ ]:
num_cols = ["Age", "Height", "Weight", "IMC", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdBu_r")
plt.title("Correlação entre variáveis numéricas")
plt.tight_layout()


## Síntese para a equipe médica

- A coorte está relativamente equilibrada nas 7 classes (272–351 pacientes).
- Histórico familiar de excesso de peso é o sinal categórico mais associado aos níveis graves.
- FAF baixo (sedentarismo) e FAVC = yes aparecem com frequência nas classes de obesidade.
- O IMC separa bem os níveis, como esperado pela definição clínica; hábitos explicam o *como intervir*.
- Não há nulos no arquivo de entrega: a pipeline pode ir direto para pré-processamento.